**This notebook runs DeSeq2 on the pv rna-seq data**
- all strains are run separately
    - sialic acid (ac and gc) vs glucose 
    - when available ac vs glucose, gc vs glucose
- results written to notebooks/results

In [ ]:
library("DESeq2")
library(apeglm)

# Read in data

**Sample metadata**

In [3]:
sample_map <- read.table("../rna_seq/samples.csv", sep=",", header=FALSE,
 col.names=c("SampleID", "path", 'genome', 'strain', 'Treatment'))
# add GC/AC sialic column
sample_map$sialic <- lapply(sample_map$Treatment, function(x) grepl("C", x))
sample_map$sialic <- factor(as.integer(sample_map$sialic))
sample_map$Treatment <- factor(sample_map$Treatment)
sample_map$Treatment <- factor(
  sample_map$Treatment,
  levels = c("Glucose", "AC", "GC")   # Glucose = reference
)
head(sample_map)

,SampleID,path,genome,strain,Treatment,sialic
,<chr>,<chr>,<chr>,<chr>,<fct>,<fct>
1,B01,vulgatus_samples/B01_S74_R1_001.fastq.gz,CL09,CL09,Glucose,0
2,F01,vulgatus_samples/F01_S78_R1_001.fastq.gz,CL09,CL09,Glucose,0
3,G01,vulgatus_samples/G01_S79_R1_001.fastq.gz,CL09,CL09,Glucose,0
4,E02,vulgatus_samples/E02_S85_R1_001.fastq.gz,CL09,CL09,AC,1
5,F02,vulgatus_samples/F02_S86_R1_001.fastq.gz,CL09,CL09,AC,1
6,G02,vulgatus_samples/G02_S87_R1_001.fastq.gz,CL09,CL09,AC,1


**counts** (double bc fadu half counts single reads)

In [4]:
load_counts <- function(samples, count_dir = "../rna_seq/vulgatus_processed") {
  counts <- Reduce(
    function(x, y) merge(x, y, by = "featureID", sort = FALSE),
    lapply(samples, function(sample) {
      df <- read.table(
        file.path(count_dir, sample, paste0(sample, ".mapped.counts.txt")),
        header = TRUE, sep = "\t"
      )
      out <- data.frame(featureID = df$featureID, counts = df$counts * 2)
      names(out)[2] <- sample
      out
    })
  )
  rownames(counts) <- counts$featureID
  counts$featureID <- NULL
  counts
}

CL09s <- subset(sample_map, strain == "CL09")$SampleID
ATCCs <- subset(sample_map, strain == "ATCC")$SampleID
BIOMLs <- subset(sample_map, strain == "BIOML")$SampleID

CL09_counts <- load_counts(CL09s)
ATCC_counts <- load_counts(ATCCs)
BIOML_counts <- load_counts(BIOMLs)

#head(CL09_counts)

In [5]:
dim(CL09_counts)
dim(ATCC_counts)
dim(BIOML_counts)

[1] 4025    9

[1] 4194    8

[1] 4194    6

# Expression per Treatment, by strain

## ATCC (expect to see nan genes come on in sialic acid)

**subset metadata, filter out low count genes**

In [ ]:
run_strain_deseq <- function(counts, strainG, full_treatment=TRUE) {
  ## pull out metadata for the strain (using sample_map from above)
  meta <- subset(sample_map, strain == strainG)
  rownames(meta) <- meta$SampleID

  ### filter low count stuff out 
  tf <- rowSums(counts > 40) >= 3
  counts_filt <- round(counts[tf, , drop = FALSE])

  ### make DeSeq objects 
  dds_sialic <- DESeqDataSetFromMatrix(countData = counts_filt, colData = meta, design = ~ sialic)
  if (full_treatment) {
    dds_treatment <- DESeqDataSetFromMatrix(countData = counts_filt, colData = meta, design = ~ Treatment)
  }

  ### run DeSeq
  dds_sialic_deseq <- DESeq(dds_sialic)
  if (full_treatment) {
  dds_treatment_deseq <- DESeq(dds_treatment)
  }
  ### apply LFC shrinkage against student's t
  sialic_shrink <- lfcShrink(dds_sialic_deseq, coef = "sialic_1_vs_0")
  if (full_treatment) {
    treatment_shrink_ac <- lfcShrink(dds_treatment_deseq, coef = "Treatment_AC_vs_Glucose")
    treatment_shrink_gc <- lfcShrink(dds_treatment_deseq, coef = "Treatment_GC_vs_Glucose")
  }

  ## and write

  write.table(
    as.data.frame(sialic_shrink),
    file = file.path('results', paste0(tolower(strainG), ".sialic.tsv")),
    sep = "\t",
    quote = FALSE,
    row.names = TRUE
  )
  if (full_treatment) {
  write.table(
    as.data.frame(treatment_shrink_ac),
    file = file.path('results', paste0(tolower(strainG), ".glucose_vs_ac.tsv")),
    sep = "\t",
    quote = FALSE,
    row.names = TRUE
  )
  write.table(
    as.data.frame(treatment_shrink_gc),
    file = file.path('results', paste0(tolower(strainG), ".glucose_vs_gc.tsv")),
    sep = "\t",
    quote = FALSE,
    row.names = TRUE
  )
  list(
    dds_sialic = dds_sialic_deseq,
    dds_treatment = dds_treatment_deseq
  )
  } else{
    list( dds_sialic = dds_sialic_deseq)
  }

}

ATCC_deseq <- run_strain_deseq(ATCC_counts, "ATCC", TRUE)
CL09_deseq <- run_strain_deseq(CL09_counts, "CL09", TRUE)
## Dont run 3-level with BIOML bc it failed in GC
BIOML_deseq <- run_strain_deseq(BIOML_counts, "BIOML", FALSE)

converting counts to integer mode

converting counts to integer mode

estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing

estimating size factors

estimating dispersions

gene-wise dispersion estimates

mean-dispersion relationship

final dispersion estimates

fitting model and testing

using 'apeglm' for LFC shrinkage. If used in published research, please cite:
    Zhu, A., Ibrahim, J.G., Love, M.I. (2018) Heavy-tailed prior distributions for
    sequence count data: removing the noise and preserving large differences.
    Bioinformatics. https://doi.org/10.1093/bioinformatics/bty895

using 'apeglm' for LFC shrinkage. If used in published research, please cite:
    Zhu, A., Ibrahim, J.G., Love, M.I. (2018) Heavy-tailed prior distributions for
    sequence count data: removing the noise and preserving large differences.
    Bioinformatics. https://doi.org/10.1093/bioinform

%optional plotting below%

In [36]:
ATCC_deseq$dds_sialic

class: DESeqDataSet 
dim: 3204 8 
metadata(1): version
assays(4): counts mu H cooks
rownames(3204): JJAGMNAF_00001 JJAGMNAF_00002 ... JJAGMNAF_04297
  JJAGMNAF_04298
rowData names(22): baseMean baseVar ... deviance maxCooks
colnames(8): A01 D01 ... C03 E03
colData names(7): SampleID path ... sialic sizeFactor